# Lesson 01d — Context Window Management
**Domain:** E-commerce · Multi-turn review Q&A  
**Dataset:** Synthetic multi-turn conversations (generated from Amazon reviews)  
**Time estimate:** ~1.5 h

## Learning objectives
- Token counting with `tiktoken` per role
- Sliding-window trim: always keep system prompt, drop oldest turns
- Trim + summarise: replace dropped messages with an LLM summary
- When to trim, when to summarise, when to upgrade to 128k context

In [ ]:
import sys, time
sys.path.insert(0, "..")

import tiktoken
from openai import OpenAI
from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODEL = "local-model"
enc   = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def llm(messages, max_tokens=200):
    resp = local_client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0.3
    )
    return resp.choices[0].message.content.strip()

print("✅ Setup complete")

## Synthetic multi-turn conversation

In [ ]:
# Generate a long multi-turn conversation about Amazon product reviews
SYSTEM_PROMPT = (
    "You are a helpful shopping assistant. Answer questions about electronics products "
    "based on customer reviews. Be concise."
)

# Build a realistic 20-turn conversation
LONG_CONVERSATION = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": "I'm looking for a good pair of wireless headphones under $100."},
    {"role": "assistant", "content": "Based on reviews, the Sony WH-CH720N and Anker Soundcore Q45 are popular choices under $100 with solid battery life and noise cancellation."},
    {"role": "user",      "content": "How is the battery life on the Sony ones?"},
    {"role": "assistant", "content": "The WH-CH720N offers up to 35 hours with ANC off, 26 hours with ANC on. Reviewers frequently mention consistent performance across the full charge."},
    {"role": "user",      "content": "What about the microphone quality for calls?"},
    {"role": "assistant", "content": "Microphone quality is average for the price — fine for calls in quiet environments but reviewers note wind noise issues outdoors."},
    {"role": "user",      "content": "Are there better options for call quality specifically?"},
    {"role": "assistant", "content": "For call quality, the Jabra Evolve2 series is highly rated but exceeds $100. Within budget, the Plantronics Voyager series has better mic reviews."},
    {"role": "user",      "content": "I also need them for exercise. Are the Sony ones sweat resistant?"},
    {"role": "assistant", "content": "No, the WH-CH720N is not officially rated for sweat or water. For workouts, consider the Jabra Elite 4 Active (IPX7) or Beats Fit Pro."},
    {"role": "user",      "content": "What's IPX7 mean exactly?"},
    {"role": "assistant", "content": "IPX7 means the device can survive immersion in up to 1 meter of water for 30 minutes. For sweat and rain during exercise, IPX4 (splash-proof) is usually sufficient."},
    {"role": "user",      "content": "OK let's switch topics. I also want a good webcam for video calls."},
    {"role": "assistant", "content": "The Logitech C920 is the most recommended webcam for video calls — 1080p, wide-angle, good low-light performance, and compatible with all platforms."},
    {"role": "user",      "content": "Any alternatives if the C920 is sold out?"},
    {"role": "assistant", "content": "The Razer Kiyo (with built-in ring light) and AVerMedia PW315 are strong alternatives. For budget options, the Logitech C270 at 720p works well for most calls."},
    {"role": "user",      "content": "I have bad lighting at my desk — does that matter?"},
    {"role": "assistant", "content": "Yes, significantly. The Razer Kiyo's ring light is worth it for poor lighting. Alternatively, a cheap LED desk lamp behind your monitor improves any webcam substantially."},
    {"role": "user",      "content": "What about 4K webcams? Are they worth it?"},
    {"role": "assistant", "content": "4K webcams like the Logitech Brio are overkill for standard video calls since most platforms cap at 1080p. Worth it only if you also do recording or streaming."},
    {"role": "user",      "content": "Let's talk keyboards. I want a mechanical keyboard for coding."},
    {"role": "assistant", "content": "For coding, the Keychron K2 or K8 are popular — hot-swappable switches, compact layout, and available in various switch types. The Logitech MX Mechanical is also well-reviewed."},
    {"role": "user",      "content": "What switch type is best for typing?"},
    {"role": "assistant", "content": "Tactile switches (Brown, Clear) are preferred for typing — they provide feedback without the loud click of blues. Linear reds are quieter but lack tactile bump."},
    {"role": "user",      "content": "I work in an office, so noise matters."},
    {"role": "assistant", "content": "For office use, go with silent linear switches (Gateron Silent Red/Black or Cherry MX Silent Red) or silent tactile switches (Gateron Silent Brown). Many reviewers specifically mention them for quiet office environments."},
]

total_tokens = sum(count_tokens(m["content"]) for m in LONG_CONVERSATION)
print(f"Conversation: {len(LONG_CONVERSATION)} messages, {total_tokens} total tokens")

---
## Exercise 1A — `count_tokens_by_role`

Write `count_tokens_by_role(messages: list[dict]) -> dict` returning
`{"system": int, "user": int, "assistant": int, "total": int}`.

In [ ]:
def count_tokens_by_role(messages: list) -> dict:
    # TODO: iterate over messages, count tokens per role
    raise NotImplementedError("Implement count_tokens_by_role()")

In [ ]:
try:
    token_counts = count_tokens_by_role(LONG_CONVERSATION)
    print("Token counts by role:")
    for role, count in token_counts.items():
        print(f"  {role:<12}: {count}")

    check([
        ("system"    in token_counts, "Has 'system' key"),
        ("user"      in token_counts, "Has 'user' key"),
        ("assistant" in token_counts, "Has 'assistant' key"),
        ("total"     in token_counts, "Has 'total' key"),
        (token_counts["total"] == sum(count_tokens(m["content"]) for m in LONG_CONVERSATION),
         "Total matches sum of all messages"),
    ], exercise_id="01d-1a")
except NotImplementedError:
    print("⚠️  Implement count_tokens_by_role() first.")

---
## Exercise 1B — `trim_conversation`

Write `trim_conversation(messages, max_tokens, system_key="system")` that:
1. Always preserves the system message
2. Drops the **oldest** user/assistant turns until total tokens ≤ `max_tokens`
3. Returns the trimmed list

**Auto-check verifies:**
- Returned conversation never exceeds `max_tokens`
- System message is always preserved
- Most recent messages are kept (oldest dropped)

In [ ]:
def trim_conversation(messages: list, max_tokens: int) -> list:
    # TODO: implement sliding-window trim
    # 1. Separate system message (first item if role=="system")
    # 2. While total tokens > max_tokens: pop the oldest non-system message
    # 3. Return [system_msg] + remaining messages
    raise NotImplementedError("Implement trim_conversation()")

In [ ]:
MAX_TOKENS = 800

try:
    trimmed = trim_conversation(LONG_CONVERSATION, MAX_TOKENS)
    trimmed_tokens = sum(count_tokens(m["content"]) for m in trimmed)
    system_preserved = trimmed[0]["role"] == "system"
    last_msg_preserved = trimmed[-1]["content"] == LONG_CONVERSATION[-1]["content"]

    print(f"Original : {len(LONG_CONVERSATION)} messages, {total_tokens} tokens")
    print(f"Trimmed  : {len(trimmed)} messages, {trimmed_tokens} tokens  (limit={MAX_TOKENS})")

    check([
        (trimmed_tokens <= MAX_TOKENS, f"Total tokens ≤ {MAX_TOKENS} ({trimmed_tokens})"),
        (system_preserved,             "System message preserved"),
        (last_msg_preserved,           "Most recent message preserved"),
        (len(trimmed) < len(LONG_CONVERSATION), "Some messages were dropped"),
    ], exercise_id="01d-1b")

except NotImplementedError:
    print("⚠️  Implement trim_conversation() first.")

---
## Exercise 1C — `trim_with_summary`

Enhance the approach: instead of silently dropping messages, replace the oldest dropped
messages with a 1-2 sentence LLM-generated summary prepended to the conversation.

Write `trim_with_summary(messages, max_tokens, llm_client) -> list`.

In [ ]:
def trim_with_summary(messages: list, max_tokens: int, llm_fn) -> list:
    """
    Like trim_conversation, but replaced dropped messages are summarised by an LLM
    and inserted as a system message: 'Earlier in this conversation: <summary>'.
    """
    # TODO: implement
    # 1. Identify which messages would be dropped by trim_conversation
    # 2. If any are dropped, call llm_fn to summarise them in 1-2 sentences
    # 3. Insert the summary as a {"role": "system", "content": "Earlier: ..."} message
    #    immediately after the original system message
    raise NotImplementedError("Implement trim_with_summary()")

In [ ]:
try:
    result = trim_with_summary(
        LONG_CONVERSATION, MAX_TOKENS,
        lambda msgs: llm(msgs, max_tokens=100)
    )
    result_tokens = sum(count_tokens(m["content"]) for m in result)
    has_summary = any("Earlier" in m["content"] for m in result if m["role"] == "system")

    print(f"trim_with_summary: {len(result)} messages, {result_tokens} tokens")
    for m in result[:4]:
        print(f"  [{m['role']:10}] {m['content'][:80]}...")

    check([
        (result_tokens <= MAX_TOKENS + 200, "Tokens within reasonable limit (summary adds some)"),
        (has_summary,                       "Summary injected as system message"),
        (result[0]["role"] == "system",     "Original system message first"),
    ], exercise_id="01d-1c")

except NotImplementedError:
    print("⚠️  Implement trim_with_summary() first.")

---
## Benchmark: trim vs summarise

Compare the two approaches on information preservation.

In [ ]:
# ── Benchmark: trim vs summarise (50-message conversation) ─────────
import time

BENCHMARK_CONV = LONG_CONVERSATION * 2 + LONG_CONVERSATION[:12]  # ~50 messages
BENCHMARK_CONV[0] = LONG_CONVERSATION[0]  # keep 1 system message

token_total = sum(count_tokens(m["content"]) for m in BENCHMARK_CONV)
print(f"Benchmark conversation: {len(BENCHMARK_CONV)} msgs, {token_total} tokens")

BUDGET = 1200

try:
    t0 = time.perf_counter()
    trimmed_b   = trim_conversation(BENCHMARK_CONV, BUDGET)
    t_trim = time.perf_counter() - t0

    t0 = time.perf_counter()
    summarised_b = trim_with_summary(BENCHMARK_CONV, BUDGET, lambda m: llm(m, max_tokens=80))
    t_summary = time.perf_counter() - t0

    tok_trim = sum(count_tokens(m["content"]) for m in trimmed_b)
    tok_sum  = sum(count_tokens(m["content"]) for m in summarised_b)

    print(f"\n{'Method':<20} {'Messages':>10} {'Tokens':>10} {'Time':>10}")
    print("-" * 54)
    print(f"{'trim':20} {len(trimmed_b):>10} {tok_trim:>10} {t_trim:>9.2f}s")
    print(f"{'trim+summary':20} {len(summarised_b):>10} {tok_sum:>10} {t_summary:>9.2f}s")
    print(f"\nCost of summarisation call: ~${(tok_sum/1000)*0.00025:.4f} (Claude Haiku)")

except NotImplementedError:
    print("⚠️  Complete both trim functions before running the benchmark.")

In [ ]:
progress()

---
## Readiness checklist — end of Module 1
- [ ] All 4 notebooks completed (01a – 01d)
- [ ] LM Studio local model tested and responding
- [ ] Async batch classification with asyncio works
- [ ] CoT + XML parsing pipeline works
- [ ] `trim_conversation()` and `trim_with_summary()` pass all checks
- [ ] Token distribution of Amazon reviews analysed (p50/p90/p99 known)